## cấu trúc
```
PromotionContext
      ↓
IDiscountStrategy
      ↓
strategy cụ thể
```
## flow
promotion
   ↓
detect rule variant
   ↓
resolve strategy
   ↓
strategy.apply()
   ↓
final discount

In [2]:
// discount_strategy_example.cs

using System;
using System.Collections.Generic;

// =========================================================
// DOMAIN MODEL
// =========================================================

public class PromotionContext
{
    public decimal OrderAmount { get; set; }

    public decimal DiscountValue { get; set; }

    public string StrategyType { get; set; }
}


// =========================================================
// STRATEGY CONTRACT
// =========================================================

public interface IDiscountStrategy
{
    decimal Apply(PromotionContext context);
}


// =========================================================
// PERCENTAGE STRATEGY
// =========================================================

public class PercentageDiscountStrategy : IDiscountStrategy
{
    public decimal Apply(PromotionContext context)
    {
        var discount = context.OrderAmount * context.DiscountValue / 100;

        return context.OrderAmount - discount;
    }
}


// =========================================================
// FIXED AMOUNT STRATEGY
// =========================================================

public class AmountDiscountStrategy : IDiscountStrategy
{
    public decimal Apply(PromotionContext context)
    {
        return context.OrderAmount - context.DiscountValue;
    }
}


// =========================================================
// BUY X GET Y STRATEGY
// =========================================================

public class BuyXGetYStrategy : IDiscountStrategy
{
    public decimal Apply(PromotionContext context)
    {
        Console.WriteLine("apply buy x get y rule");

        return context.OrderAmount;
    }
}


// =========================================================
// STRATEGY RESOLVER
// =========================================================

public static class DiscountStrategyResolver
{
    private static readonly Dictionary<string, IDiscountStrategy> strategies =
        new Dictionary<string, IDiscountStrategy>()
    {
        { "PERCENT", new PercentageDiscountStrategy() },
        { "AMOUNT", new AmountDiscountStrategy() },
        { "BUY_X_GET_Y", new BuyXGetYStrategy() }
    };

    public static IDiscountStrategy Resolve(string type)
    {
        if (!strategies.ContainsKey(type))
            throw new Exception("strategy not found");

        return strategies[type];
    }
}


// =========================================================
// DEMO
// =========================================================

var promotion = new PromotionContext
{
    OrderAmount = 1000,
    DiscountValue = 10,
    StrategyType = "PERCENT"
};


// resolve strategy
var strategy =
    DiscountStrategyResolver.Resolve(promotion.StrategyType);


// apply rule
var finalAmount =
    strategy.Apply(promotion);


Console.WriteLine($"final amount: {finalAmount}");


final amount: 900
